# Dual Approach for Bermudan Option Pricing: Alfonsi's Algorithm

In the following, we discuss and implement the algorithm proposed by Alfonsi for pricing 
Bermudan options using a **pure dual approach**. The section is organized as follows:

1. We introduce the **dual representation** of the optimal stopping problem.
2. We show how this leads to a **backward minimization problem**.
3. We discuss the **strict convexification technique** and the theoretical result that 
justifies it (Theorem 1.4 in Alfonsi's article).
4. We describe the **practical framework**: the approximation space for martingale 
increments, the choice of $\phi$, and the resulting least squares problem.
5. Finally, we present the **Monte Carlo estimator** and the training/pricing split.

---

## 1. The Optimal Stopping Problem and Its Dual Representation

The classical discrete-time optimal stopping problem is:

$$
U_n = \sup_{\tau \in \mathcal{T}_{n,N}} \mathbb{E}[Z_\tau \mid \mathcal{F}_n],
$$

where $\mathcal{T}_{n,N}$ denotes the set of stopping times taking values in $\{n, \ldots, N\}$,
and $Z_\tau$ represents the discounted payoff at the stopping time $\tau$.

It can be shown that $U_n$ admits an equivalent **dual representation**:

$$
U_n = \inf_{M \in \mathbb{H}^p} \mathbb{E}\left[\max_{n \leq j \leq N} 
\{Z_j - (M_j - M_n)\} \,\bigg|\, \mathcal{F}_n \right],
$$

where the infimum is taken over all martingales $M \in \mathbb{H}^p$. Intuitively, the dual 
approach seeks the martingale that best approximates the Snell envelope from above.

---

## 2. Decomposition into a Backward Minimization Problem

Rewriting the global minimization problem in terms of **martingale increments** 
$\Delta M_{n+1} = M_{n+1} - M_n$, one obtains:

$$
U_0 = \inf_{M \in \mathbb{H}^p} \mathbb{E}\left[\max_{0 \leq j \leq N} 
\{Z_j - (M_j - M_0)\}\right]
$$

$$
= \mathbb{E}[Z_N] + \inf_{\Delta M_1 \in \mathcal{H}_1^p, \ldots, \Delta M_N \in \mathcal{H}_N^p} 
\sum_{n=0}^{N-1} \mathbb{E}\left[\left(Z_n + \Delta M_{n+1} - 
\max_{n+1 \leq j \leq N}\left\{Z_j - \sum_{i=n+2}^{j} \Delta M_i\right\}\right)_+\right].
$$

This decomposition is key: it allows us to split the global minimization into a **sequence 
of local minimization problems**, solved **backward in time** from $n = N-1$ to $n = 0$. 
At each step $n$, one minimizes over $\Delta M_{n+1} \in \mathcal{H}_{n+1}^p$:

$$
\inf_{\Delta M_{n+1} \in \mathcal{H}_{n+1}^p} \mathbb{E}\left[\left(Z_n + \Delta M_{n+1} - 
\max_{n+1 \leq j \leq N}\left\{Z_j - \sum_{i=n+2}^{j} \Delta M_i\right\}\right)_+\right],
$$

treating the future increments $\Delta M_{i}$ for $i \geq n+2$ as already determined from 
previous backward steps.

---

## 3. Strict Convexification

A key difficulty is that the positive part function $x \mapsto x_+$ is **convex but not 
strictly convex**, which leads to a **non-uniqueness issue** for the minimizer.

To overcome this, one replaces $x_+$ with a **strictly convex function** $\phi$. This is 
justified by **Theorem 1.4** in Alfonsi's article, which guarantees that if $\phi$ is 
strictly convex, then:

- the minimizer is **unique**,
- it coincides with the martingale appearing in the **Doob–Meyer decomposition** of $U_n$.

Hence the strict convexification does not alter the solution, but restores uniqueness and 
makes the problem tractable.

---

## 4. Practical Framework

### 4.1 Approximation Space for Martingale Increments

We approximate the space of martingale increments via **linear combinations** of basis 
elements of the form:

$$
X^{p,k}_{t_{i,j}} - X^{p,k}_{t_{i,j-1}} = u^p_{i,j-1}(S_{t_{i,j-1}})
\left(\mathcal{A}^k_{t_{i,j}} - \mathcal{A}^k_{t_{i,j-1}}\right),
$$

where:
- the index $i = 1, \ldots, N$ labels the **exercise dates** of the Bermudan option,
- the index $j = 1, \ldots, \bar{N}$ subdivides each interval $[t_{i-1}, t_i]$ into 
$\bar{N}$ **sub-intervals**,
- $u^p_{i,j-1}(S_{t_{i,j-1}})$ is a family of functions mapping the asset space to 
$\mathbb{R}$, evaluated at the previous sub-interval date,
- $\mathcal{A}^k$ are discounted asset processes whose increments provide the martingale 
structure.

The martingale increment over the $i$-th exercise period is then:

$$
\Delta M_{i+1} = \sum_{j=1}^{\bar{N}} \sum_{p,k} \alpha^{p,k}_{i,j} 
\left(X^{p,k}_{t_{i,j}} - X^{p,k}_{t_{i,j-1}}\right),
$$

where the coefficients $\alpha^{p,k}_{i,j} \in \mathbb{R}$ are the **parameters to optimize**.

### 4.2 Choice of $\phi$ and Least Squares Reduction

We choose $\phi(x) = x^2$, which is strictly convex. Under this choice, the backward 
minimization problem at each stage reduces to a **classical least squares problem**, 
solvable via standard linear algebra techniques.

---

## 5. Monte Carlo Estimator and Training/Pricing Split

Once the optimal coefficients $\{\alpha^{p,k}_{i,j}\}$ have been computed, the price 
estimator is:

$$
\hat{U}_0^Q = \frac{1}{Q} \sum_{q=1}^{Q} \max_{0 \leq j \leq N} 
\left\{ \hat{Z}_j^q - \sum_{i=1}^{j} \alpha_i^Q \cdot \Delta \hat{X}_i^q \right\}.
$$

As in the **Longstaff–Schwartz algorithm**, to avoid **overfitting** (since the 
coefficients $\alpha$ are estimated on the same paths used for pricing), we split 
the simulation into two independent phases:

- **Training phase**: simulate a first set of paths and solve the backward least squares 
problems to compute the optimal coefficients $\alpha^Q$.
- **Pricing phase**: simulate a fresh, independent set of paths and use the estimated 
coefficients to evaluate the Monte Carlo estimator $\hat{U}_0^Q$.

This separation ensures that the estimator remains unbiased and avoids the lookahead bias 
that would arise from using the same paths for both calibration and pricing.

In [1]:
from code_utils import *
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from scipy.stats import norm

In [8]:
# Parameters

N = 10 			# Number of exercising dates
N_bar = 10 		# Number of intervals between two exercising dates 
K = 100 		# Strike price
S0 = 100 		# Initial price
r = 0.06 		# Risk-free rate
sigma = 0.4 	# Volatility
T = 0.5 		# Maturity
seed = 42	 	# Seed for reproducibility
paths = 10**5 	# Number of Monte Carlo paths
P = 10 			# Number of basis functions 
L = N_bar * P 

exercise_time = np.linspace(0,T, N + 1) 
exercise_indices = range(0, N_bar * N + 1, N_bar)
dense_grid = np.linspace(0,T, N_bar * N + 1) 
dt_dense = dense_grid[1] - dense_grid[0]

coefficients = np.zeros((N, L)) # Since the backward induction goes from N - 1 to 0.
discount_factor = np.exp(-r * dense_grid)
discount_factor_exercise_time = np.exp(-r * exercise_time)

# Training phase
asset = S0 * simulate_gbm(r, sigma, T, dt_dense, paths, seed)
discounted_asset = asset * discount_factor[:, np.newaxis] # Assets are evaluated in the dense grid
discounted_payoff = np.maximum(K - asset[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis] # Payoffs are evaluated only at exercise times

# Computing basis function 
# Cumulative distribution function of log-normal distribution 
log_asset = np.log(asset)
sigma_local = np.std(log_asset, axis=1)
mu_local = np.mean(log_asset, axis=1)
log_asset_normalized = (log_asset - mu_local[:, np.newaxis]) / sigma_local[:, np.newaxis]
phi_matrix = norm.cdf(log_asset_normalized)
partition = np.linspace(0, 1, P + 1)

# Local basis function 
basis_function = np.zeros((N_bar * N + 1, paths, P))
for p in range(P):
    mask = (phi_matrix >= partition[p]) & (phi_matrix < partition[p+1])
    basis_function[:, :, p] = mask.astype(int)


difference_discounted_asset = np.diff(discounted_asset, axis=0)
coefficients = np.zeros((N, L))
martingale_increments = np.zeros((N, paths))

# Backward induction 
for i in range(N-1, -1, -1):
    # Discounted payoffs from i+1 onwards
    Z = discounted_payoff[i+1:, :]
    # Cumulative martingale already computed for periods i+1 to N-1
    if i + 1 < N:
        sum_to_remove = np.vstack([np.zeros((1, paths)), np.cumsum(martingale_increments[i+1:, :], axis=0)])
    else:
        sum_to_remove = np.zeros((Z.shape[0], paths))

    vector_to_maximize = np.max(Z - sum_to_remove, axis=0)  
    martingale_increment_i = np.zeros(paths)

    for j in range(N_bar):
        # Discounted asset increment at t_{i,j}
        delta_A = difference_discounted_asset[i * N_bar + j, :]  
        indicators = basis_function[i * N_bar + j , :, :]  
        delta_X_all = indicators * delta_A[:, np.newaxis] # paths x P basis functions   
        # Mean over paths for each basis function
        A_all = np.mean(delta_X_all**2, axis=0)                
        
        # b_all[p] = mean over paths of delta_X_all[:,p] * vector_to_maximize
        b_all = np.mean(delta_X_all * vector_to_maximize[:, np.newaxis], axis=0) 
        
        # alpha[p] = b[p] / A[p] per ogni p
        alpha_all = np.zeros(P)
        
        A_all_reg = A_all + 1e-8 # Avoid division by zero
        alpha_all = b_all / A_all_reg
        
        coefficients[i, j*P:(j+1)*P] = alpha_all
        
        # incremento martingala = sum_p alpha[p] * delta_X[:,p]
        # = delta_X_all @ alpha_all
        martingale_increment_i + = delta_X_all @ alpha_all      
    
    martingale_increments[i, :] = martingale_increment_i

# Pricing phase : independent paths to avoid overfitting 
pricing_paths = 10**6
asset_pricing = S0 * simulate_gbm(r, sigma, T, dt_dense, pricing_paths, seed+1)
discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
discounted_payoff_pricing = np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis]

log_asset_pricing = np.log(asset_pricing)
mu_pricing = np.mean(log_asset_pricing, axis=1)
sigma_pricing = np.std(log_asset_pricing, axis=1)
log_asset_pricing_normalized = (log_asset_pricing - mu_pricing[:, np.newaxis]) / sigma_pricing[:, np.newaxis]
phi_matrix_pricing = norm.cdf(log_asset_pricing_normalized)
basis_function_pricing = np.zeros((N_bar * N + 1, pricing_paths, P))
for p in range(P):
    mask = (phi_matrix_pricing >= partition[p]) & (phi_matrix_pricing < partition[p+1])
    basis_function_pricing[:, :, p] = mask.astype(int)

difference_discounted_asset_pricing = np.diff(discounted_asset_pricing, axis=0)
martingale_increments_pricing = np.zeros((N, pricing_paths))

for i in range(N-1, -1, -1):
    martingale_increment_i = np.zeros(pricing_paths)
    for j in range(N_bar):
        delta_A = difference_discounted_asset_pricing[i * N_bar + j, :]  
        indicators = basis_function_pricing[i * N_bar + j , :, :]         
        delta_X_all = indicators * delta_A[:, np.newaxis]                    
        alpha_all = coefficients[i, j*P:(j+1)*P]                             
        martingale_increment_i += delta_X_all @ alpha_all                    
    martingale_increments_pricing[i, :] = martingale_increment_i

max_per_path = np.max(discounted_payoff_pricing - np.vstack([np.zeros((1, pricing_paths)), np.cumsum(martingale_increments_pricing, axis=0)]), axis=0)
option_price_upper_hat = np.mean(max_per_path)
undercertainty_upper_hat = antithetic_variates(max_per_path, pricing_paths)

cumulative_martingale = np.cumsum(martingale_increments, axis=0) 
option_price_upper = np.mean(np.max(discounted_payoff - np.vstack([np.zeros((1, paths)), cumulative_martingale]), axis=0))
undercertainty_upper = antithetic_variates(np.max(discounted_payoff - np.vstack([np.zeros((1, paths)), cumulative_martingale]), axis=0), paths)


# Print results : price and its uncertainty 
print(f"Option price upper bound with new samples: {option_price_upper_hat:.4f} ± {undercertainty_upper_hat:.4f}")
print(f"Option price upper bound without new samples: {option_price_upper:.4f} ± {undercertainty_upper:.4f}")


Option price upper bound with new samples: 10.2647 ± 0.0021
Option price upper bound without new samples: 10.1923 ± 0.0065


In [20]:
# We could add the calculation with the Longstaff-Schwartz method to obtain a lower bound.
# Lower bound using Longstaff-Schwartz method
npaths_train= 10**5 # Number of paths for estimating the coefficients of the regression 
npaths_pricing = 10**6 # Number of paths for estimating the price of the option

dt = exercise_time[1] - exercise_time[0]
# Paths simulation 
train_set = simulate_gbm(r, sigma, T, dt, npaths_train, seed=seed)
test_set = simulate_gbm( r, sigma, T, dt, npaths_pricing, seed = seed + 1)

# Applying transformation
train_log = transformation(train_set)
test_log = transformation(test_set)

# Precomputing the basis for the regression since after transformation the input doesn't depend on S0
max_deg = 6
basis_train = np.ones((N-1,npaths_train, max_deg + 1))	
basis_pricing = np.ones((N-1, npaths_pricing, max_deg + 1))
for d in range(1, max_deg + 1):
	basis_train[:, :, d] = train_log[1:N,:] * basis_train[:, :, d-1]
	basis_pricing[:, :, d] = test_log[1:N,:] * basis_pricing[:, :, d-1]


ls_american, _, _, _, _ = lower_bound_ls(K, S0, r, 0, T, dt, sigma, train_set, test_set, train_log, test_log, discount_factor_exercise_time, npaths_train, npaths_pricing, max_deg, basis_train, basis_pricing, seed)
print(f"Standard Longstaff-Schwartz price: {ls_american}")


Standard Longstaff-Schwartz price: 9.905336161769338


In [4]:
# Parameters
N = 10 			# Number of exercising dates
N_bar = 10 		# Number of intervals between two exercising dates 
K = 100 		# Strike price
S0 = 100 		# Initial price
r = 0.06 		# Risk-free rate
delta = 0.00 	# Dividend yield
sigma = 0.4 	# Volatility
T = 0.5 		# Maturity
seed = 42	 	# Seed for reproducibility
paths = 10**5 	# Number of Monte Carlo paths

eta= 9      # max polynomial degree  →  P = eta + 1 monomials (d=1)
P= eta + 1  # number of basis functions = {1, x, x^2, ..., x^eta}
L = N_bar * P

exercise_time              = np.linspace(0, T, N + 1)
exercise_indices           = range(0, N_bar * N + 1, N_bar)
dense_grid                 = np.linspace(0, T, N_bar * N + 1)
dt_dense                   = dense_grid[1] - dense_grid[0]

discount_factor                  = np.exp(-r * dense_grid)
discount_factor_exercise_time    = np.exp(-r * exercise_time)


# Polynomial normalisation  φ_{i,j}: ℝ₊ → [0,1]
#   φ(x) = (x − C⁻) / (C⁺ − C⁻)

def compute_phi(asset, t_grid, S0, r, delta, sigma):

    drift = (r - delta - 0.5 * sigma**2) * t_grid          # (n_steps+1,)
    spread = 4.0 * np.sqrt(np.maximum(t_grid, 0)) * sigma   # (n_steps+1,)

    C_minus = S0 * np.exp(drift - spread)   # (n_steps+1,)
    C_plus  = S0 * np.exp(drift + spread)   # (n_steps+1,)
    width   = C_plus - C_minus              # (n_steps+1,)

    # Avoid division by zero at t=0
    safe_width = np.where(width > 1e-12, width, 1.0)
    phi = (asset - C_minus[:, np.newaxis]) / safe_width[:, np.newaxis]

    # At t=0 set phi = 0.5 otherwise division by zero would occur
    phi[t_grid == 0.0, :] = 0.5
    # np.clip(phi, 0.0, 1.0)
    return phi

def compute_basis(phi, P):

    n_t, n_paths = phi.shape
    basis = np.empty((n_t, n_paths, P))
    for p in range(P):
        basis[:, :, p] = phi ** p
    return basis

# Training phase

asset = S0 * simulate_gbm(r, sigma, T, dt_dense, paths, seed)
discounted_asset = asset * discount_factor[:, np.newaxis]
discounted_payoff = (np.maximum(K - asset[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

phi_train = compute_phi(asset, dense_grid, S0, r, delta, sigma)
basis_function = compute_basis(phi_train, P)

difference_discounted_asset = np.diff(discounted_asset, axis=0)
coefficients = np.zeros((N, L))
martingale_increments = np.zeros((N, paths))

# Backward induction

# For the polynomial basis u^p are NOT mutually orthogonal w.r.t. (ΔA)²,
# so we must solve the full P×P linear system per sub-interval j:

for i in range(N - 1, -1, -1):
    Z = discounted_payoff[i + 1:, :]

    if i + 1 < N:
        sum_to_remove = np.vstack([ np.zeros((1, paths)), np.cumsum(martingale_increments[i + 1:, :], axis=0) ])
    else:
        sum_to_remove = np.zeros((Z.shape[0], paths))

    vector_to_maximize = np.max(Z - sum_to_remove, axis=0)   # (paths,)
    martingale_increment_i = np.zeros(paths)

    for j in range(N_bar):
        delta_A   = difference_discounted_asset[i * N_bar + j, :]   # (paths,)
        u_vals    = basis_function[i * N_bar + j, :, :]              # (paths, P)

        # Elementary martingale increments
        delta_X = u_vals * delta_A[:, np.newaxis]                    # (paths, P)

        # Matrix
        A_mat = (delta_X.T @ delta_X) / paths                       # (P, P)

        # Coefficient
        b_vec = (delta_X.T @ vector_to_maximize) / paths             # (P,)

        # Regularisation to keep the system well-conditioned
        A_reg     = A_mat + 1e-8 * np.eye(P)
        alpha_all = np.linalg.solve(A_reg, b_vec)                    # (P,)

        coefficients[i, j * P:(j + 1) * P] = alpha_all
        martingale_increment_i += delta_X @ alpha_all
    martingale_increments[i, :] = martingale_increment_i

# Pricing phase 
# Upper-bound estimate on TRAINING paths
cumulative_martingale = np.cumsum(martingale_increments, axis=0)
option_price_upper  = np.mean( np.max(discounted_payoff - np.vstack([np.zeros((1, paths)), cumulative_martingale]), axis=0) )
undercertainty_upper  = antithetic_variates( np.max(discounted_payoff - np.vstack([np.zeros((1, paths)), cumulative_martingale]), axis=0), paths )

# Pricing phase  (independent paths)

pricing_paths = 10**6

asset_pricing  = S0 * simulate_gbm(r, sigma, T, dt_dense, pricing_paths, seed + 1)
discounted_asset_pricing = asset_pricing * discount_factor[:, np.newaxis]
discounted_payoff_pricing = (np.maximum(K - asset_pricing[exercise_indices, :], 0) * discount_factor_exercise_time[:, np.newaxis])

phi_pricing = compute_phi(asset_pricing, dense_grid, S0, r, delta, sigma)
basis_function_pricing = compute_basis(phi_pricing, P)

difference_discounted_asset_pricing = np.diff(discounted_asset_pricing, axis=0)
martingale_increments_pricing = np.zeros((N, pricing_paths))

for i in range(N - 1, -1, -1):
    martingale_increment_i = np.zeros(pricing_paths)
    for j in range(N_bar):
        delta_A = difference_discounted_asset_pricing[i * N_bar + j, :]
        u_vals = basis_function_pricing[i * N_bar + j, :, :]
        delta_X = u_vals * delta_A[:, np.newaxis]
        alpha_all = coefficients[i, j * P:(j + 1) * P]
        martingale_increment_i += delta_X @ alpha_all
        
    martingale_increments_pricing[i, :] = martingale_increment_i

max_per_path = np.max( discounted_payoff_pricing - np.vstack([np.zeros((1, pricing_paths)), np.cumsum(martingale_increments_pricing, axis=0)]), axis=0 )
option_price_upper_hat = np.mean(max_per_path)
undercertainty_upper_hat = antithetic_variates(max_per_path, pricing_paths)

# Results

print(f"Polynomial basis  (eta={eta}, P={P} monomials per sub-interval)")
print(f"Option price upper bound  [new samples] : {option_price_upper_hat:.4f} ± {undercertainty_upper_hat:.4f}")
print(f"Option price upper bound  [train paths] : {option_price_upper:.4f}    ± {undercertainty_upper:.4f}")

Polynomial basis  (eta=9, P=10 monomials per sub-interval)
Option price upper bound  [new samples] : 10.4960 ± 0.0037
Option price upper bound  [train paths] : 10.4498    ± 0.0057
